In [ ]:
import pandas as pd
import os
import numpy as np
from Bio import SeqIO
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## part1：multi-TIS result

In [ ]:
df_multiTIS = pd.read_csv("table.tisseq_multiTIS.gar.csv")
target_cols = ['ORF_ID','ORF_type', 'gene_id', 'chrom','ORF_gstart','ORF_gstop','strand', 'AAseq']
df_multiTIS_simple = df_multiTIS[target_cols]

In [ ]:
df_multiTIS_filter = df_multiTIS_simple[df_multiTIS_simple["ORF_type"].isin(["annotated", "Overlap_uORF",  "Overlap_dORF","internal"])]
gstart_counts = df_multiTIS_filter.groupby("gene_id")["ORF_gstart"].nunique()
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
df_multi_gstart = df_multiTIS_filter[df_multiTIS_filter["gene_id"].isin(genes_multi_gstart)]

print(f"CDS 相关multi-TIS 基因数目: {len(genes_multi_gstart)}")
print(f"CDS 相关multi-TIS 事件数目: {len(df_multi_gstart)}")
df_multi_gstart.to_csv("table.CDS_related_multiTIS.csv", index=False)

In [ ]:
annotated_genes = (
    df_multi_gstart
    .groupby("gene_id")["ORF_type"]
    .apply(lambda x: (x == "annotated").any())
)

annotated_genes = annotated_genes[annotated_genes].index

df_multiTIS_final = df_multi_gstart[
    df_multi_gstart["gene_id"].isin(annotated_genes)
].copy()

In [ ]:
orf_type_counts = df_multiTIS_final["ORF_type"].value_counts()
print(orf_type_counts)

In [ ]:
orf_types_of_interest = ["annotated", "Overlap_uORF", "Overlap_dORF","internal"]
df_type2 = df_multiTIS_final[df_multiTIS_final['ORF_type'].isin(orf_types_of_interest)].copy()

type2_inframe_events = []    
type2_outframe_events = []  

genes_inframe = set()
genes_outframe = set()

# =========================================
# 逐基因分析（5′端 annotated 作为 reference）
# =========================================
for gene, df_gene in df_type2.groupby('gene_id'):

    strand = df_gene['strand'].iloc[0]

    # 5′ 端 annotated 作为 reference
    df_anno = df_gene[df_gene['ORF_type'] == 'annotated']
    if len(df_anno) == 0:
        continue

    if strand == '+':
        ref_row = df_anno.sort_values('ORF_gstart').iloc[0]
    else:
        ref_row = df_anno.sort_values('ORF_gstart', ascending=False).iloc[0]

    ref_orf_id = ref_row['ORF_ID']
    ref_start  = ref_row['ORF_gstart']
    ref_seq    = ref_row['AAseq']
    ref_len    = len(ref_seq)

    # 遍历该基因所有候选 ORF
    for idx, row in df_gene.iterrows():

        orf_id = row['ORF_ID']
        start  = row['ORF_gstart']
        seq    = row['AAseq']
        var_len = len(seq)

        if orf_id == ref_orf_id:
            continue  # 跳过 reference 自身

        diff_nt = start - ref_start

        # =====================================================
        # 框内起始 → N端延伸 或 N端截短
        # =====================================================
        if abs(diff_nt) % 3 == 0:

            delta_aa = diff_nt // 3

            # ---------------------------
            # 情况 1：N 端延伸
            # ---------------------------
            if delta_aa < 0:
                ext_len = abs(delta_aa)
                extension_seq = seq[:ext_len]  
                truncation_seq = ""           
                change_type = "N-terminal_extension"

            # ---------------------------
            # 情况 2：N 端截短
            # ---------------------------
            elif delta_aa > 0:
                ext_len = delta_aa
                extension_seq = ""              
                truncation_seq = ref_seq[:ext_len]  
                change_type = "N-terminal_truncation"

            # ---------------------------
            # 情况 3：delta_aa == 0（起始完全一致）
            # ---------------------------
            else:
                continue

            type2_inframe_events.append({
                'gene_id': gene,
                'strand': strand,
                'ref_ORF_ID': ref_orf_id,
                'var_ORF_ID': orf_id,
                'ref_start': ref_start,
                'var_start': start,
                'delta_nt': diff_nt,
                'delta_aa': delta_aa,
                'ORF_type': row['ORF_type'],
                'change_type': change_type,
                'extension_len_aa': abs(delta_aa),
                'extension_seq': extension_seq,
                'truncation_seq': truncation_seq,
                'ref_AAseq_full': ref_seq,
                'var_AAseq_full': seq
            })

            genes_inframe.add(gene)

        # =====================================================
        # 框外起始 → 全新蛋白
        # =====================================================
        else:
            type2_outframe_events.append({
                'gene_id': gene,
                'strand': strand,
                'ref_ORF_ID': ref_orf_id,
                'var_ORF_ID': orf_id,
                'ref_start': ref_start,
                'var_start': start,
                'delta_nt': diff_nt,
                'ORF_type': row['ORF_type'],
                'effect': 'novel_protein',
                'var_AAseq_full': seq
            })

            genes_outframe.add(gene)

df_inframe  = pd.DataFrame(type2_inframe_events)
df_outframe = pd.DataFrame(type2_outframe_events)

# =========================================
# 统计输出
# =========================================
print("\n✅ 最终统计结果（reference = 5′ annotated）\n")

print("【类型1：框内起始（N端截短 / 延伸）】")
print(f"事件数: {len(df_inframe)}")
print(f"基因数: {len(genes_inframe)}\n")

print("【类型2：框外起始（全新蛋白）】")
print(f"事件数: {len(df_outframe)}")
print(f"基因数: {len(genes_outframe)}")

# =========================================
# 导出 FASTA
# =========================================
os.makedirs("functional_fasta", exist_ok=True)

# ✅ 5.1 N端【延伸】片段 FASTA
with open("functional_fasta/inframe_N_terminal_extensions.fasta", "w") as f:
    for _, row in df_inframe.iterrows():
        if row['change_type'] == "N-terminal_extension":
            header = f">{row['gene_id']}|{row['var_ORF_ID']}|extension|len={row['extension_len_aa']}"
            f.write(header + "\n")
            f.write(row['extension_seq'] + "\n")

# ✅ 5.2 N端【截短】丢失片段 FASTA（来自 reference）
with open("functional_fasta/inframe_N_terminal_truncations.fasta", "w") as f:
    for _, row in df_inframe.iterrows():
        if row['change_type'] == "N-terminal_truncation":
            header = f">{row['gene_id']}|{row['var_ORF_ID']}|truncation|len={row['extension_len_aa']}"
            f.write(header + "\n")
            f.write(row['truncation_seq'] + "\n")

# ✅ 5.3 框外：完整新蛋白 FASTA
with open("functional_fasta/outframe_novel_proteins.fasta", "w") as f:
    for _, row in df_outframe.iterrows():
        header = f">{row['gene_id']}|{row['var_ORF_ID']}|novel_protein"
        f.write(header + "\n")
        f.write(row['var_AAseq_full'] + "\n")

# =========================================
# 保存完整结果
# =========================================
df_inframe.to_csv("functional_fasta/type1_inframe_events.csv", index=False)
df_outframe.to_csv("functional_fasta/type2_outframe_events.csv", index=False)

## part2：function analysis

In [ ]:
# ==============================
# 文件路径
# ==============================
fasta_trunc  = "functional_anno/inframe_N_terminal_truncations.filtered.fasta"
fasta_ext    = "functional_anno/inframe_N_terminal_extensions.filtered.fasta"
fasta_novel  = "functional_anno/outframe_novel_proteins.filtered.fasta"
interpro_tsv = "functional_anno/step1_InterProScan/interpro.merg.tsv"  
signalp_tsv = "functional_anno/step2_SignalP/signalp.merg.tsv" 

In [ ]:
# ==============================
# 读取 FASTA 文件
# ==============================
def read_fasta_length(fasta_file, type_name):
    seqs = list(SeqIO.parse(fasta_file, "fasta"))
    df = pd.DataFrame({
        "gene_id": [rec.id.split(".")[0] for rec in seqs],
        "length_change": [len(rec.seq) for rec in seqs],  # Type3 存储的就是截短/延长长度
        "type": type_name
    })
    return df

df_trunc = read_fasta_length(fasta_trunc, "N_terminal_truncation")
df_ext   = read_fasta_length(fasta_ext, "N_terminal_extension")
df_novel = read_fasta_length(fasta_novel, "Novel_protein")

# 合并
df_all = pd.concat([df_trunc, df_ext, df_novel], ignore_index=True)

In [ ]:
# ==============================
# 可视化 N端长度变化
# ==============================
df_type3 = df_all[df_all["type"].isin(["N_terminal_truncation","N_terminal_extension"])]

plt.figure(figsize=(5,3))
sns.histplot(df_type3["length_change"], bins=30, kde=True, color="salmon", alpha=0.7)
plt.xlabel("N-terminal length change (aa)")
plt.ylabel("Gene count")
plt.title("type1 protein length changes")
plt.xlim(0, 800)  # 设置横轴范围为0-1000
plt.tight_layout()
plt.savefig("plot_N_terminal_change_boxplo.pdf", format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ==============================
# 读入 InterProScan 结果
# ==============================
cols = [ "protein_id", "md5", "length", "analysis", "signature_accession", "signature_desc", "start", "end", "evalue", "status", "date",
    "ipr_id", "ipr_desc", "go", "pathway"]
df_ipr = pd.read_csv(interpro_tsv, sep="\t", header=None, names=cols)

# 提取 gene_id & type
df_ipr["gene_id"] = df_ipr["protein_id"].str.split("|").str[0]
df_ipr["type"] = df_ipr["protein_id"].str.split("|").str[2]   # extension / truncation / outframe

# 只保留 Type3 + Type4
df_ipr = df_ipr[df_ipr["type"].isin(["extension", "truncation", "novel_protein"])]

# ==============================
# 只保留高可信结构域来源（防止重复统计）
# ==============================
keep_db = ["Pfam", "SMART", "CDD"]

df_ipr_filt = df_ipr[
    (df_ipr["analysis"].isin(keep_db)) &
    (df_ipr["ipr_id"].notna())
].copy()

# 只保留真正“结构域级注释”
df_ipr_filt = df_ipr_filt[df_ipr_filt["signature_accession"].notna()]

def classify_gain_loss(x):
    if x == "extension":
        return "Gain"
    elif x == "truncation":
        return "Loss"
    else:
        return "Novel"

df_ipr_filt["Change"] = df_ipr_filt["type"].apply(classify_gain_loss)
df_ipr_filt['signature_desc'] = (
    df_ipr_filt['signature_desc'].str.replace("Leucine Rich Repeat", "Leucine rich repeat", regex=False)
)
df_ipr_filt['signature_desc'] = (
    df_ipr_filt['signature_desc'].str.replace("Leucine Rich repeat", "Leucine rich repeat", regex=False)
)

In [ ]:
df_type3_domain = (
    df_ipr_filt
    .groupby(["signature_desc", "Change"])
    #.groupby(["ipr_id", "Change"])
    .size()
    .reset_index(name="Count")
    .rename(columns={"signature_desc": "Domain"})
)

# 只保留 Gain / Loss
df_type3_domain = df_type3_domain[df_type3_domain["Change"].isin(["Gain", "Loss"])]

In [ ]:
# ==============================
# 读入 SignalP 结果
# ==============================
df_sp = pd.read_csv( signalp_tsv, sep="\t", comment="#",
    names=["protein_id", "Prediction", "OTHER", "SP", "CS"]
)

df_sp["gene_id"] = df_sp["protein_id"].str.split("_").str[0]
df_sp["type"] = df_sp["protein_id"].str.extract(r"(extension|truncation|novel_protein)")[0]

df_sp["SP_flag"] = df_sp["Prediction"].str.contains("SP", na=False)

In [ ]:
df_sp_merge = df_sp.merge(
    df_ipr_filt[["protein_id", "Change"]],
    on="protein_id",
    how="left"
)

df_sp_merge.groupby("Change")["SP_flag"].mean()

In [ ]:
df_gain = df_type3_domain[df_type3_domain["Change"] == "Gain"].sort_values("Count", ascending=False).head(10)
df_loss = df_type3_domain[df_type3_domain["Change"] == "Loss"].sort_values("Count", ascending=False).head(10)
# 统一 Domain 顺序
all_domains = list(set(df_gain["Domain"]) | set(df_loss["Domain"]))
df_plot = pd.DataFrame({"Domain": all_domains})
df_plot = df_plot.merge(df_gain[["Domain", "Count"]].rename(columns={"Count":"Gain"}), on="Domain", how="left")
df_plot = df_plot.merge(df_loss[["Domain", "Count"]].rename(columns={"Count":"Loss"}), on="Domain", how="left")
df_plot = df_plot.fillna(0)
# 先创建一列 Total
df_plot["Total"] = df_plot["Gain"] + df_plot["Loss"]
# 按 Total 排序
df_plot = df_plot.sort_values("Total", ascending=True)

In [ ]:
# ----------------------
# 统计 SP Gain / Loss
# ----------------------
sp_gain = df_sp[(df_sp["type"]=="extension") & (df_sp["SP_flag"]==True)].shape[0]
sp_loss = df_sp[(df_sp["type"]=="truncation") & (df_sp["SP_flag"]==True)].shape[0]

# ----------------------
# 在 df_plot 末尾添加 SP
# ----------------------
df_plot_sp = df_plot.copy()
df_plot_sp = df_plot_sp.append({
    "Domain":"SignalP",
    "Gain": sp_gain,
    "Loss": sp_loss,
    "Total": sp_gain + sp_loss
}, ignore_index=True)

# 按 Total 排序
df_plot_sp = df_plot_sp.sort_values("Total", ascending=True)

# ----------------------
# 绘图
# ----------------------
fig, ax = plt.subplots(figsize=(6, 5))

# 左边 Gain（取负数）
ax.barh(df_plot_sp["Domain"], -df_plot_sp["Gain"], color="#FF7F0E", alpha=0.8)
# 右边 Loss
ax.barh(df_plot_sp["Domain"], df_plot_sp["Loss"], color="#1F77B4", alpha=0.8)

# 在条形末端添加数字
for idx, row in df_plot_sp.iterrows():
    if row["Gain"] > 0:
        ax.text(-row["Gain"]-0.5, idx, int(row["Gain"]), va='center', ha='right', fontsize=8, color='k')
    if row["Loss"] > 0:
        ax.text(row["Loss"]+0.5, idx, int(row["Loss"]), va='center', ha='left', fontsize=8, color='k')

ax.axvline(0, color="k", linewidth=0.8)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([abs(int(x)) for x in xticks])

ax.set_xlabel("Gene count")
ax.set_title("Type3 protein domain Gain/Loss (with SignalP)")

gain_patch = mpatches.Patch(color="#FF7F0E", label="Gain")
loss_patch = mpatches.Patch(color="#1F77B4", label="Loss")
ax.legend(handles=[gain_patch, loss_patch])

plt.tight_layout()
plt.savefig("plot_N_terminal_domain.pdf", format='pdf', bbox_inches='tight')
plt.show()
plt.close()
